# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# # Install uv (fast Python package/environment manager)
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create fresh virtual environment
# !uv venv .venv --seed

# # Upgrade pip tooling
# !.venv/bin/python -m pip install --upgrade pip setuptools wheel

# # Install Blackwell / RTX 5090 compatible PyTorch nightly (CUDA 12.8)
# !.venv/bin/python -m pip install --pre torch torchvision torchaudio \
#     --index-url https://download.pytorch.org/whl/nightly/cu128

# # Install vLLM + remaining dependencies
# !.venv/bin/python -m pip install \
#     vllm==0.11.0 \
#     transformers==4.57.0 \
#     sympy \
#     numpy \
#     tqdm \
#     bitsandbytes \
#     antlr4-python3-runtime==4.11.1 \
#     ipykernel \
#     jupyter

# !.venv/bin/python -m pip install hf-transfer

# # Register Jupyter kernel
# !.venv/bin/python -m ipykernel install \
#     --user \
#     --name cse151b \
#     --display-name "Python (cse151b)"

# print("✅ Environment installation complete.")
# print("⚠️ IMPORTANT: Restart the notebook kernel now.")
# print("Then select:")
# print("Kernel -> Change Kernel -> Python (cse151b)")

### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [ ]:
import sys
import torch
import transformers
import vllm

print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU capability:", torch.cuda.get_device_capability(0))
print("Transformers:", transformers.__version__)
print("vLLM:", vllm.__version__)

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

### Generate with Transformers (for Datahub)

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

## 8. Summary

Print accuracy broken down by question type.

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [ ]:
import os, re, json, csv, time, math
from collections import Counter
from fractions import Fraction

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["HF_HUB_DISABLE_XET"] = "1"

MODEL_ID   = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH  = "data/private.jsonl"
OUT_PATH   = "results/submission.csv"

N_SAMPLES   = 1
TEMPERATURE = 0.6     
TOP_P        = 0.95
MAX_TOKENS  = 32768
TP_SIZE     = 1       

os.makedirs("results", exist_ok=True)
print("✅ Config ready for single RTX 5090.")

In [ ]:
def extract_boxed(text: str) -> str | None:
    """Brace-balanced extraction of the LAST \\boxed{...}. Handles nested braces."""
    results = []
    pos = 0
    while True:
        start = text.find(r'\boxed{', pos)
        if start == -1:
            break
        depth = 0
        content_start = start + len(r'\boxed{')
        end = -1
        for j in range(content_start - 1, len(text)):
            if text[j] == '{':
                depth += 1
            elif text[j] == '}':
                depth -= 1
                if depth == 0:
                    end = j
                    break
        if end != -1:
            results.append(text[content_start:end].strip())
            pos = end + 1
        else:
            results.append(text[content_start:].strip())
            break
    return results[-1] if results else None


def latex_to_numeric(s: str) -> str | None:
    """Resolves common LaTeX numeric expressions to plain numbers."""
    s = s.strip()
    m = re.fullmatch(r'\\frac\{(-?\d+)\}\{(-?\d+)\}', s)
    if m:
        try:
            f = Fraction(int(m.group(1)), int(m.group(2)))
            v = float(f)
            return str(int(v)) if v == int(v) else str(f)
        except Exception:
            pass
    m = re.fullmatch(r'\\sqrt\{(\d+)\}', s)
    if m:
        v = math.sqrt(int(m.group(1)))
        return str(int(v)) if v == int(v) else f"{v:.6g}"
    m = re.fullmatch(r'(-?[\d.]+)\s*\\times\s*10\^\{(-?\d+)\}', s)
    if m:
        try:
            v = float(m.group(1)) * 10 ** int(m.group(2))
            return str(int(v)) if v == int(v) else f"{v:.6g}"
        except Exception:
            pass
    return None


def normalize(ans: str) -> str:
    """Canonical form for stable majority-vote comparison without regex crash risks."""
    if not isinstance(ans, str):
        return ""
    
    s = ans.strip()
    s = s.replace('\\$', '').strip('$').strip()
    s = s.replace('\\,', '').replace('\\ ', '').replace('\\!', '').strip()

    resolved = latex_to_numeric(s)
    if resolved is not None:
        return resolved

    m = re.fullmatch(r'(-?\d+)\s*/\s*(-?\d+)', s)
    if m:
        try:
            f = Fraction(int(m.group(1)), int(m.group(2)))
            v = float(f)
            # Catch infinities from massive fractions
            if math.isinf(v) or math.isnan(v):
                return str(f)
            return str(int(v)) if v == int(v) else str(f)
        except Exception:
            pass

    try:
        v = float(s.replace(',', ''))
        # Catch infinities explicitly so they don't hit the int() cast
        if math.isinf(v) or math.isnan(v):
            return s.lower()
        return str(int(v)) if v == int(v) else f"{v:.6g}"
    except (ValueError, OverflowError): # FIXED: Now catches the infinity overflow
        pass

    if re.fullmatch(r'[a-jA-J]', s):
        return s.upper()

    if ',' in s:
        return ', '.join(normalize(p.strip()) for p in s.split(','))

    return s.lower()



def fallback_mcq(text: str) -> str:
    tail = text[-400:]
    m = re.search(r'\b(?:answer\s+is\s+|option\s+|choice\s+)?([A-J])[\.\)]?\s*$', tail, re.IGNORECASE)
    if m:
        return m.group(1).upper()
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1].upper() if letters else "A"


def fallback_numeric(text: str, ans_count: int) -> str:
    tail = text[-500:]
    nums = re.findall(r'-?\d+(?:\.\d+)?', tail)
    if nums:
        if ans_count > 1 and len(nums) >= ans_count:
            return ", ".join(nums[-ans_count:])
        return nums[-1]
    return ", ".join(["0"] * max(1, ans_count))


def extract_answer(text: str, is_mcq: bool, ans_count: int) -> str:
    boxed = extract_boxed(text)
    if boxed:
        return boxed
    return fallback_mcq(text) if is_mcq else fallback_numeric(text, ans_count)


def majority_vote(answers: list[str]) -> str:
    normed = [normalize(a) for a in answers if a]
    if not normed:
        return answers[0] if answers else "0"
    winner = Counter(normed).most_common(1)[0][0]
    for a in answers:
        if normalize(a) == winner:
            return a
    return answers[0]

print("✅ Safe Utilities ready.")

In [ ]:

SYSTEM_PROMPT_FREE = (
    "You are an expert mathematical reasoner.\n\n"
    "ANSWER FORMAT RULES:\n"
    "1. Put ALL answers inside a single \\boxed{}, comma-separated, in the exact order "
    "the [ANS] placeholders appear. Example (3 placeholders): \\boxed{41, 35, 16}.\n"
    "2. EMBEDDED CHOICE: If an [ANS] slot is immediately followed by labeled options "
    "(e.g. '[ANS] A. Reject  B. Fail to reject', or '[ANS] A. Yes  B. No'), "
    "output ONLY the capital letter for that slot — not the option text. "
    "Mixed example: \\boxed{1.96, A, 0.032, B}.\n"
    "3. Give EXACT answers (fractions, radicals, expressions) unless rounding is explicitly "
    "requested, then use the precision specified. Reduce all fractions to lowest terms.\n"
    "4. For answers 'in terms of' a variable, use the exact variable name; do not evaluate.\n"
    "5. Do NOT include units in \\boxed{} unless the problem explicitly requires them.\n"
    "6. Output \\boxed{} immediately after your reasoning. Nothing after the box."
)
 
SYSTEM_PROMPT_FREE_OLYMPIAD = (
    "You are an expert mathematical reasoner solving a competition-level problem.\n\n"
    "ANSWER FORMAT RULES:\n"
    "1. Place your single final answer in \\boxed{}.\n"
    "2. Give exact answers (integers, reduced fractions, radicals) unless told to round.\n"
    "3. Commit to one approach and execute it fully. If you find a flaw, correct inline "
    "and continue — do not restart from scratch.\n"
    "4. Output \\boxed{} immediately after your reasoning. Nothing after the box."
)
 
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematical reasoner.\n\n"
    "ANSWER FORMAT RULES:\n"
    "1. Output ONLY the capital letter of the correct option inside \\boxed{}. "
    "Options run A through J. Example: \\boxed{G}.\n"
    "2. Write the letter only — do NOT reproduce the option text.\n"
    "3. Treat ALL options as valid candidates, including 'Unable to determine', "
    "'None of the above', and 'Unchanged'.\n"
    "4. Output \\boxed{} immediately after your reasoning. Nothing after the box."
)

print("✅ Prompts compiled.")

In [ ]:
from transformers import AutoTokenizer

data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} problems.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer ready.")

In [ ]:
def get_system_prompt(question: str, options) -> str:
    if options is not None:
        return SYSTEM_PROMPT_MCQ
    elif question.count("[ANS]") == 0:
        return SYSTEM_PROMPT_FREE_OLYMPIAD
    else:
        return SYSTEM_PROMPT_FREE

prompts = []
for item in data:
    options  = item.get("options")
    qid      = item["id"]
    question = item["question"]
    system   = get_system_prompt(question, options)

    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user      = f"Problem:\n{question}\n\nAnswer Choices:\n{opts_text}\n\nSelect the correct letter."
    else:
        user = f"Problem:\n{question}\n\nSolve completely and provide your final answer in \\boxed{{}}."

    try:
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )

    prompts.append(prompt)

print(f"✅ Built {len(prompts)} prompts.")

In [ ]:
from vllm import LLM, SamplingParams

print("Loading model onto RTX 5090...")
llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=True,        
    gpu_memory_utilization=0.90,       
    max_model_len=32768,              
    trust_remote_code=True,
    tensor_parallel_size=TP_SIZE,     
)
print("✅ Model loaded successfully on single GPU instance.")

In [ ]:
sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    stop=["<|im_end|>", "<|endoftext|>"],
)

t0 = time.time()
print(f"Generating {N_SAMPLES} traces × {len(prompts)} problems…")
outputs = llm.generate(prompts, sampling_params=sampling_params)
print(f"✅ Generation done in {(time.time()-t0)/60:.1f} min.")

In [ ]:
missing_boxes = 0

with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"], quoting=csv.QUOTE_ALL)
    writer.writeheader()

    for i, out in enumerate(outputs):
        item      = data[i]
        is_mcq    = bool(item.get("options"))
        ans_count = 1 if is_mcq else item["question"].count("[ANS]")

        texts   = [o.text.strip() for o in out.outputs]
        answers = [extract_answer(t, is_mcq, ans_count) for t in texts]

        for t in texts:
            if extract_boxed(t) is None:
                missing_boxes += 1

        final_ans = majority_vote(answers)

        # Pick the representative trace from the winning cluster
        rep_text = texts[0]
        for t, a in zip(texts, answers):
            if normalize(a) == normalize(final_ans):
                rep_text = t
                break

        response = rep_text + f"\n\n\\boxed{{{final_ans}}}"
        writer.writerow({"id": item["id"], "response": response})

print(f"✅ Saved → {OUT_PATH}")
print(f"    Fallbacks (no \\boxed): {missing_boxes} / {len(outputs) * N_SAMPLES} traces")

In [ ]:
import torch

elapsed = (time.time() - t0) / 60
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "RTX 5090"

with open("README.md", "w") as f:
    f.write(
        f"# CSE 151B Submission\n\n"
        f"**Model:** {MODEL_ID}  \n"
        f"**GPU:** {gpu}  \n"
        f"**N_SAMPLES (self-consistency):** {N_SAMPLES}  \n"
        f"**MAX_TOKENS:** {MAX_TOKENS}  \n"
        f"**Inference time:** ~{elapsed:.1f} min\n\n"
        f"## Reproduce\n"
        f"```python\n"
        f"# Run cells 1–9 in order\n"
        f"```\n"
    )

print(f"✅ README.md written. Total elapsed: {elapsed:.1f} min | GPU: {gpu}")